# Options from Zero · Episode 18: Hedging in practice: implied vs realised volatility

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. The experiment · 4. No hedge · 5. Hedge weekly, hedge daily · 6. The twist · 7. Why: theta earned, gamma paid · 8. This episode's question · 9. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
n_paths = 4000   # simulated years per experiment

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 18,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

In [ ]:
out["trade"] = {"spot": S, "strike": K, "years": T, "rate_pct": r * 100, "vol_pct": vol * 100}

Part 2's additions: Greeks, simulation, hedging, implied volatility, Black and Bachelier (`episodes/_shared/options_part2.py`):

In [ ]:
# Part 2 additions to options.py: the Greeks, simulation, a hedging simulator, implied volatility, and Black's and
# Bachelier's formulas on a forward (for FX and interest-rate options). Written out by hand and copied verbatim into
# each Part 2 notebook; the notebooks check them against QuantLib. Needs options.py (norm_cdf, black_scholes),
# math and numpy as np.

def norm_pdf(x):
    """The bell curve's height at x (standard normal density)."""
    return math.exp(-x * x / 2) / math.sqrt(2 * math.pi)

def norm_cdf_v(x):
    """norm_cdf for a whole numpy array at once."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x, dtype=float) / math.sqrt(2)))

def bs_greeks(s, k, t, r, vol):
    """Price and Greeks of a European call on a share with no dividends.
    delta per $1 of share price, gamma per $1 squared, vega per 1.00 of volatility (100 vol points),
    theta per year (negative: value lost as time passes), rho per 1.00 of interest rate."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    disc = math.exp(-r * t)
    return {"price": s * norm_cdf(d1) - k * disc * norm_cdf(d2),
            "delta": norm_cdf(d1),
            "gamma": norm_pdf(d1) / (s * vol * math.sqrt(t)),
            "vega": s * norm_pdf(d1) * math.sqrt(t),
            "theta": -s * norm_pdf(d1) * vol / (2 * math.sqrt(t)) - r * k * disc * norm_cdf(d2),
            "rho": k * t * disc * norm_cdf(d2)}

def call_delta(s, k, t, r, vol):
    """Black-Scholes delta N(d1) for an array of share prices."""
    d1 = (np.log(np.asarray(s, dtype=float) / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    return norm_cdf_v(d1)

def simulate_paths(s, t, drift, vol, steps, n_paths, rng):
    """Share price paths: shape (n_paths, steps + 1), starting at s, lognormal steps with the given drift."""
    dt = t / steps
    z = rng.standard_normal((n_paths, steps))
    log_steps = (drift - vol ** 2 / 2) * dt + vol * math.sqrt(dt) * z
    return s * np.exp(np.concatenate([np.zeros((n_paths, 1)), np.cumsum(log_steps, axis=1)], axis=1))

def monte_carlo_call(s, k, t, r, vol, n, rng):
    """Price = the average discounted payoff over n risk-neutral outcomes, with its standard error."""
    z = rng.standard_normal(n)
    s_t = s * np.exp((r - vol ** 2 / 2) * t + vol * math.sqrt(t) * z)
    pv = math.exp(-r * t) * np.maximum(s_t - k, 0.0)
    return float(pv.mean()), float(pv.std(ddof=1) / math.sqrt(n))

def hedge_short_call(s, k, t, r, vol_implied, vol_realised, steps, n_paths, rng, hedge=True):
    """Sell the call at vol_implied and, if hedge, rebalance to delta shares (computed at vol_implied) `steps` times.
    The share moves with vol_realised and grows at r. Cash earns r. Returns each path's P&L at expiry in today's dollars."""
    dt = t / steps
    price = s * np.ones(n_paths)
    held = call_delta(price, k, t, r, vol_implied) if hedge else np.zeros(n_paths)
    cash = black_scholes(s, k, t, r, vol_implied) - held * price      # premium received, shares bought
    for i in range(1, steps + 1):
        price = price * np.exp((r - vol_realised ** 2 / 2) * dt + vol_realised * math.sqrt(dt) * rng.standard_normal(n_paths))
        cash = cash * math.exp(r * dt)
        if hedge and i < steps:
            new = call_delta(price, k, t - i * dt, r, vol_implied)
            cash = cash - (new - held) * price                        # buy (or sell) the difference
            held = new
    pnl = cash + held * price - np.maximum(price - k, 0.0)           # sell the shares, pay the option's payoff
    return pnl * math.exp(-r * t)

def implied_vol(price, s, k, t, r, lo=0.0001, hi=3.0, tol=1e-12):
    """Run Black-Scholes backwards by bisection: halve [lo, hi] until the price fits. Returns (vol, halvings)."""
    n = 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if black_scholes(s, k, t, r, mid) < price:
            lo = mid
        else:
            hi = mid
        n += 1
    return (lo + hi) / 2, n

def black76(forward, strike, t, vol, df, call=True):
    """Black's formula on a forward: df x [F N(d1) - K N(d2)] for a call. FX options (Garman-Kohlhagen) and caplets."""
    sd = vol * math.sqrt(t)
    d1 = (math.log(forward / strike) + sd ** 2 / 2) / sd
    d2 = d1 - sd
    if call:
        return df * (forward * norm_cdf(d1) - strike * norm_cdf(d2))
    return df * (strike * norm_cdf(-d2) - forward * norm_cdf(-d1))

def bachelier(forward, strike, t, normal_vol, df, call=True):
    """Bachelier's formula: the forward moves by normal_vol x sqrt(t) in absolute terms (it can go below zero)."""
    sd = normal_vol * math.sqrt(t)
    d = (forward - strike) / sd
    if call:
        return df * ((forward - strike) * norm_cdf(d) + sd * norm_pdf(d))
    return df * ((strike - forward) * norm_cdf(-d) + sd * norm_pdf(d))

## 2. Last episode's question

An error bar of ±0.40 with 1,000 paths. Ten times more accurate needs $10^2 = 100$ times the paths: 100,000.

In [ ]:
out["answer"] = {"paths": 1_000, "se": 0.40, "target": 0.04, "needed": 1_000 * (0.40 / 0.04) ** 2}
out["answer"]

## 3. The experiment

We **sell** the one-year \$105 call at 20% volatility and collect its Black-Scholes price. Then we hedge: hold delta shares (computed at 20%), rebalanced every week or every day, borrowing and lending cash at the risk-free rate. Meanwhile the share moves with its *realised* volatility, which may or may not be 20%. At expiry we pay the option's payoff and count the profit, in today's dollars.

To keep the story about volatility, the share grows at the risk-free rate in these simulations (with a hedge, its growth rate barely matters; without one, it would shift the average).

In [ ]:
premium = black_scholes(S, K, T, r, vol)
bins_wide = np.arange(-30.0, 10.0001, 1.0)
bins = np.arange(-7.0, 5.0001, 0.25)
def hist(p, b):
    c, _ = np.histogram(np.clip(p, b[0] + 1e-9, b[-1] - 1e-9), bins=b)
    return (c / len(p)).round(5).tolist()
runs = {}
def run(label, steps, realised, hedge=True):
    p = hedge_short_call(S, K, T, r, vol, realised, steps, n_paths, np.random.default_rng(18), hedge=hedge)
    runs[label] = p
    return {"mean": float(p.mean()), "sd": float(p.std()), "p05": float(np.percentile(p, 5)),
            "se": float(p.std() / math.sqrt(len(p))), "worst": float(p.min())}
out["premium"] = premium
out["n_paths"] = n_paths
out["tail_pct"] = 5
out["gap_pts"] = 5
out["x_wide"] = ((bins_wide[:-1] + bins_wide[1:]) / 2).tolist()
out["x"] = ((bins[:-1] + bins[1:]) / 2).round(2).tolist()
out["bin"] = 0.25
out["bin_wide"] = 1.0
out["ticks_wide"] = [{"x": float(v), "label": f"{'+' if v > 0 else ('−' if v < 0 else '')}${abs(v):.0f}"} for v in (-30, -20, -10, 0, 10)]
out["ticks"] = [{"x": float(v), "label": f"{'+' if v > 0 else ('−' if v < 0 else '')}${abs(v):.0f}"} for v in (-6, -4, -2, 0, 2, 4)]
premium

## 4. No hedge

Sell the call at 20% and do nothing. Most of the time the premium is profit, but the losses in the tail are many times the premium.

In [ ]:
out["none"] = run("none", 252, 0.20, hedge=False)
out["none"]["hist"] = hist(runs["none"], bins_wide)
out["none"]["p05_abs"] = abs(out["none"]["p05"])
out["none"]["worst_abs"] = abs(out["none"]["worst"])
out["none"]["share_profit"] = float((runs["none"] > 0).mean())
{k: v for k, v in out["none"].items() if k != "hist"}

## 5. Hedge weekly, hedge daily

Realised volatility equals the 20% we sold at. Hedging shrinks the spread; the average stays near zero, because at a fair volatility the premium is exactly the expected cost of hedging. Hedging 252 times instead of 52 should shrink the spread by about $\sqrt{252/52} \approx 2.2$.

In [ ]:
out["weekly"] = run("weekly", 52, 0.20)
out["daily"] = run("daily", 252, 0.20)
for k in ("weekly", "daily"):
    out[k]["hist"] = hist(runs[k], bins)
out["sd_ratio"] = out["weekly"]["sd"] / out["daily"]["sd"]
out["sd_ratio_theory"] = math.sqrt(252 / 52)
assert abs(out["daily"]["mean"]) < 2 * out["daily"]["se"] + 0.02
assert abs(out["sd_ratio"] / out["sd_ratio_theory"] - 1) < 0.15
{k: out[k]["sd"] for k in ("none", "weekly", "daily")} | {"ratio": out["sd_ratio"]}

## 6. The twist

Same daily hedge, but the share turns out calmer (15% realised) or wilder (25%) than the 20% we sold at. The spread stays small, but the average moves: a hedged option is a bet on volatility.

In [ ]:
out["calm"] = run("calm", 252, 0.15)
out["wild"] = run("wild", 252, 0.25)
for k in ("calm", "wild"):
    out[k]["hist"] = hist(runs[k], bins)
out["wild"]["mean_abs"] = abs(out["wild"]["mean"])
out["calm"]["realised_pct"], out["wild"]["realised_pct"] = 15, 25
g = bs_greeks(S, K, T, r, vol)
out["vega_5"] = g["vega"] * 0.05
out["vega"] = g["vega"]
out["calm_profit_share"] = float((runs["calm"] > 0).mean())
out["wild_loss_share"] = float((runs["wild"] < 0).mean())
{k: out[k]["mean"] for k in ("calm", "wild")} | {"vega x 5 pts": out["vega_5"]}

## 7. Why: theta earned, gamma paid

Each day the hedged seller collects theta and pays for gamma (Episode 15). With Black-Scholes at the implied volatility, one day's P&L of the hedged short call is about

$$\tfrac12\,\Gamma S^2\Big(\sigma_{\text{implied}}^2\,dt - \big(\tfrac{dS}{S}\big)^2\Big)$$

positive on quiet days, negative on wild ones. Summed over the year it averages to about vega $\times(\sigma_{\text{implied}} - \sigma_{\text{realised}})$. Check path by path: the simulated P&L against this sum.

In [ ]:
def gamma_sum(steps, realised, n, seed):
    rng_ = np.random.default_rng(seed)
    dt = T / steps
    s = np.full(n, S)
    total = np.zeros(n)
    for i in range(steps):
        tau = T - i * dt
        d1 = (np.log(s / K) + (r + vol ** 2 / 2) * tau) / (vol * math.sqrt(tau))
        gam = np.exp(-d1 ** 2 / 2) / math.sqrt(2 * math.pi) / (s * vol * math.sqrt(tau))
        s_new = s * np.exp((r - realised ** 2 / 2) * dt + realised * math.sqrt(dt) * rng_.standard_normal(n))
        total += 0.5 * gam * s ** 2 * (vol ** 2 * dt - (s_new / s - 1) ** 2) * math.exp(-r * i * dt)
        s = s_new
    return total
approx = gamma_sum(252, 0.25, n_paths, 18)
corr = float(np.corrcoef(approx, runs["wild"])[0, 1])
out["decomposition"] = {"corr": corr, "approx_mean": float(approx.mean())}
assert corr > 0.9
out["decomposition"]

## 8. This episode's question

You **bought** options at 20% and hedged them; realised volatility was 25%. The buyer is long gamma: wilder moves pay more than the theta cost. Profit, of about vega × 5 points.

In [ ]:
out["question"] = {"implied_pct": 20, "realised_pct": 25, "profit_approx": out["vega_5"]}
out["question"]

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")